Импортируем все необходимые библиотеки

In [14]:
# Основные библиотеки
import pandas as pd
import io, zipfile, urllib.request

# Загрузка данных
from ucimlrepo import fetch_ucirepo 
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score, roc_auc_score, accuracy_score

Загружаем данные

In [15]:
url = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"
outer = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read()))
inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
df = pd.read_csv(inner.open("bank-additional/bank-additional-full.csv"), sep=";")

In [16]:
# bank_marketing = fetch_ucirepo(id=222)

# X = bank_marketing.data.features
# y = bank_marketing.data.targets

Задаем категориальные фичи

In [17]:
CATS = ["job", "marital", "education", "default", "housing",
        "loan", "contact", "poutcome", "month", "day_of_week"]

Очищаем данные

In [18]:
def prepare(df):
    d = df.copy()
    d["y"] = d["y"].eq("yes").astype(int)

    for c in CATS:
        d[c] = d[c].fillna("unknown").astype(str)

    # pdays = дней с прошлого контакта; -1 (bank-full) и 999 (bank-additional)
    # означают "прошлого контакта не было", а не реальное число дней
    never = d["pdays"].isin([-1, 999])
    d["was_contacted"] = (~never).astype(int)
    d["pdays"] = d["pdays"].where(~never)

    return d.drop(columns=["duration"])


In [19]:
data = prepare(df)
X_all, y_all = data.drop(columns="y"), data["y"]

X_tr, X_rest, y_tr, y_rest = train_test_split(
    X_all, y_all, test_size=0.30, stratify=y_all, random_state=42
)
X_va, X_te, y_va, y_te = train_test_split(
    X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=42
)

In [ ]:
model = CatBoostClassifier(
    iterations=3000, loss_function="Logloss", eval_metric="PRAUC",
    random_seed=42, early_stopping_rounds=150, cat_features=CATS,
    verbose=100, learning_rate=0.05, depth=6,
    one_hot_max_size=20
)

model.fit(X_tr, y_tr, eval_set=(X_va, y_va))
p_va, p_te = model.predict_proba(X_va)[:, 1], model.predict_proba(X_te)[:, 1]

prec, rec, thr = precision_recall_curve(y_va, p_va)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
best_thr = thr[f1s[:-1].argmax()]


0:	learn: 0.4068057	test: 0.4139874	best: 0.4139874 (0)	total: 6.38ms	remaining: 19.1s
100:	learn: 0.4999643	test: 0.4877495	best: 0.4877495 (100)	total: 584ms	remaining: 16.8s
200:	learn: 0.5363493	test: 0.4881882	best: 0.4890818 (123)	total: 1.16s	remaining: 16.2s
300:	learn: 0.5672931	test: 0.4909144	best: 0.4925560 (258)	total: 1.73s	remaining: 15.5s
400:	learn: 0.5923172	test: 0.4895027	best: 0.4925560 (258)	total: 2.33s	remaining: 15.1s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.4925560156
bestIteration = 258

Shrink model to first 259 iterations.


Результаты

In [42]:
print(f"======== Метрики на тестовой выборке ========")
print(f"Порог: {best_thr:.3f}")
print(f"F1: {f1_score(y_te, p_te > best_thr):.3f}")
print(f"PR-AUC: {average_precision_score(y_te, p_te):.3f}")
print(f"ROC-AUC: {roc_auc_score(y_te, p_te):.3f}")

======== Метрики на тестовой выборке ========
Порог: 0.204
F1: 0.526
PR-AUC: 0.481
ROC-AUC: 0.810
